# 11 — Metrics & dashboard

CallMetricsAccumulator, MetricsStore, dashboard SSE, CSV/JSON export, pricing, basic auth.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises `CallMetricsAccumulator`, `MetricsStore`, and export helpers.


### CallMetricsAccumulator


In [ ]:
import { CallMetricsAccumulator } from "getpatter";
await cell('metrics_accumulator', { tier: 1, env }, () => {
  const acc = new CallMetricsAccumulator({
    callId: 'call-demo-001',
    providerMode: 'openai_realtime',
    telephonyProvider: 'twilio',
  });
  acc.startTurn();
  acc.recordSttComplete('What is the weather?', 2.1);
  acc.recordLlmFirstToken();
  acc.recordTtsFirstByte();
  const tm = acc.recordTurnComplete('The weather is sunny.');
  console.log(`turn_index: ${tm.turnIndex}  tts_chars: ${tm.ttsCharacters}`);
  console.log(`latency.total_ms: ${tm.latency.totalMs?.toFixed(0)}ms`);
});


### MetricsStore


In [ ]:
import { MetricsStore } from "getpatter";
await cell('metrics_store', { tier: 1, env }, () => {
  const store = new MetricsStore({ maxCalls: 100 });
  store.recordCallStart({ callId: 'c001', from: '+15555550100', to: '+15555550200', direction: 'inbound', carrier: 'twilio', provider: 'openai_realtime' });
  store.recordCallEnd({ callId: 'c001', duration: 45, costTotal: 0.08 });
  const agg = store.getAggregates();
  console.log(`total calls: ${agg.totalCalls}  avg_duration: ${agg.avgDuration}s`);
});


### Export


In [ ]:
import { MetricsStore, callsToCsv, callsToJson } from "getpatter";
await cell('export', { tier: 1, env }, () => {
  const store = new MetricsStore();
  store.recordCallStart({ callId: 'c001', from: '+15555550100', to: '+15555550200', direction: 'inbound', carrier: 'twilio', provider: 'openai_realtime' });
  store.recordCallEnd({ callId: 'c001', duration: 45, costTotal: 0.08 });
  const calls = store.getCalls();
  const csv  = callsToCsv(calls);
  const json = callsToJson(calls);
  console.log(`CSV header: ${csv.split('\n')[0]}`);
  console.log(`JSON: ${json.slice(0, 60)}...`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real call and inspects the `MetricsStore` after it ends. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, () => {
  console.log(`  carrier:  Twilio ${env.twilioNumber}  →  ${env.targetNumber}`);
  console.log('  metrics:  will inspect MetricsStore after call ends');
  console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
});


### Live call + metrics inspection *(T4)*


In [ ]:
import { Patter, Twilio, OpenAIRealtime, MetricsStore } from "getpatter";
await cell('live_metrics_call', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, async () => {
  const store = new MetricsStore({ maxCalls: 50 });
  const p = new Patter({
    carrier: new Twilio({ accountSid: env.twilioSid, authToken: env.twilioToken }),
    phoneNumber: env.twilioNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Metrics demo. Greet and hang up.',
    engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
  });
  try {
    await p.call(env.targetNumber, { agent, firstMessage: 'Metrics demo.', ringTimeout: env.maxCallSeconds });
    const agg = store.getAggregates();
    console.log(`Calls in store: ${agg.totalCalls}`);
    console.log(`Avg duration:   ${agg.avgDuration}s`);
    console.log('✓ Metrics call completed');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
